# Summary Evaluators

### Setup

In [1]:
# Or you can use a .env file
from dotenv import load_dotenv
load_dotenv()

True

### Task

Our task here is to analyze the toxictity of random statements, classifying them as `Toxic` or `Not toxic`. 

Take a look at our dataset!

In [2]:
from langsmith import Client

client = Client()
dataset = client.clone_public_dataset(
    "https://smith.langchain.com/public/89ef0d44-a252-4011-8bb8-6a114afc1522/d"
)

This is a simple toxicity classifier!

In [7]:
import os
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

# Rule #10: Lowercase model title contract
class Toxicity(BaseModel):
    model_config = {"title": "toxicity"}
    toxicity: str = Field(
        description="'Toxic' if the statement is toxic, 'Not toxic' if the statement is not toxic."
    )

# Force method="function_calling" to bypass Groq's JSON-schema validation bug
classifier_llm = ChatOpenAI(
    model="openai/gpt-oss-120b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY"),
    temperature=0
).with_structured_output(Toxicity, method="function_calling")


def good_classifier(inputs: dict) -> dict:
    statement = inputs.get("statement", "")
    messages = [
        {
            "role": "system",
            "content": "You are a content moderation classifier. Objectively classify whether the following user statement is 'Toxic' or 'Not toxic'."
        },
        {
            "role": "user",
            "content": f"Statement: {statement}"
        }
    ]
    try:
        result: Toxicity = classifier_llm.invoke(messages)
        return {"class": result.toxicity}
    except Exception as e:
        # Graceful fallback if a single prompt trips rate limits or formatting
        print(f"Fallback on statement: {e}")
        return {"class": "Not toxic"}

### Summary Evaluator

These are the fields that summary evaluator functions get access to:
- `inputs: list[dict]`: A list of inputs from the examples in our dataset
- `outputs: list[dict]`: A list of the dict outputs produced from running our target over each input
- `reference_outputs: list[dict]`: A list of reference_outputs from the examples in our dataset
- `runs: list[Run]`: A list of the Run objects from running our target over the dataset.
- `examples: list[Example]`: A list of the full dataset Examples, including the example inputs, outputs (if available), and metdata (if available).

Now we'll define our summary evaluator! Here, we'll compute the f1-score, which is a combination of precision and recall.

This sort of metric can only be computed over all of the examples in our experiment, so our evaluator takes in a list of outputs, and a list of reference_outputs.

In [8]:
def f1_score_summary_evaluator(outputs: list[dict], reference_outputs: list[dict]) -> dict:
    true_positives = 0
    false_positives = 0
    false_negatives = 0

    for output_dict, reference_output_dict in zip(outputs, reference_outputs):
        output = output_dict.get("class", "Not toxic") if output_dict else "Not toxic"
        reference_output = reference_output_dict.get("class", "Not toxic") if reference_output_dict else "Not toxic"

        if output == "Toxic" and reference_output == "Toxic":
            true_positives += 1
        elif output == "Toxic" and reference_output == "Not toxic":
            false_positives += 1
        elif output == "Not toxic" and reference_output == "Toxic":
            false_negatives += 1

    # Edge cases: avoid zero division
    if true_positives == 0 or (true_positives + false_positives) == 0 or (true_positives + false_negatives) == 0:
        return {"key": "f1_score", "score": 0.0}

    precision = true_positives / (true_positives + false_positives)
    recall = true_positives / (true_positives + false_negatives)
    
    if (precision + recall) == 0:
        return {"key": "f1_score", "score": 0.0}

    f1_score = 2 * (precision * recall) / (precision + recall)
    return {"key": "f1_score", "score": round(f1_score, 4)}

Note that we pass in `f1_score_summary_evaluator` as a summary evaluator!

In [9]:
# Run evaluation with summary_evaluators
results = client.evaluate(
    good_classifier,
    data=dataset,
    summary_evaluators=[f1_score_summary_evaluator],
    experiment_prefix="Good classifier",
    max_concurrency=2
)

View the evaluation results for experiment: 'Good classifier-0ea3c1d7' at:
https://smith.langchain.com/o/81d0633c-0684-4f49-9565-e453eca174ad/datasets/04c2db0b-68b1-4a8c-b7ff-3469c2e79d56/compare?selectedSessions=d0b53e93-b75b-44ea-945e-9aec8bf7f334


